# Chapter 9 — Ontologies and Natural Languages
### Notebook 1 · Verbalisation, and the inverse that grades it

*Book reference: Section 9.2*

Rendering an axiom as a sentence is easy. Rendering it so that the sentence can be turned back into the same axiom is what makes the output trustworthy — and it is why the language has to be *controlled*.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch09_toolkit as ch9
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

## 1. Templates

One template per construct. This is a **controlled** natural language: a deliberately small fragment of English, chosen so that parsing it back is possible at all.

In [ ]:
for operator, template in ch9.TEMPLATES['en'].items():
    print(f'  {operator:12s} {template}')

In [ ]:
for axiom in ch9.SAMPLE_AXIOMS:
    print(f'{str(axiom):48s} -> {ch9.verbalise(axiom)}')

## 2. The inverse

`parse_cnl` runs the templates backwards. Note the ordering constraint in the grammar: `Every X eats only Y` must be tried **before** the bare `Every X is a Y`, or the general pattern swallows the specific one. Ordering the rules is the entire difficulty of parsing a CNL.

In [ ]:
sentence = 'Every lion eats at least one herbivore.'
print('sentence :', sentence)
print('recovered:', ch9.parse_cnl(sentence))
print()
print('free prose does not parse:', ch9.parse_cnl('Lions tend to eat herbivores.'))

## 3. The round trip is the grader

Verbalise, parse, compare. No gold labels, no annotation, no judge — and the answer is exact.

In [ ]:
rows = [ch9.round_trips(a) for a in ch9.SAMPLE_AXIOMS]
print(pd.DataFrame(rows)[['sentence', 'recovered', 'ok']].to_string(index=False))
assert all(r['ok'] for r in rows)

## 4. What the round trip cannot see

Here is the most important cell in the chapter. Look at the disjointness sentence.

In [ ]:
bad = ch9.verbalise(ch9.Axiom('Plant', 'disjoint', 'Animal'))
print('sentence      :', bad)
print('round-trips OK:', ch9.round_trips(ch9.Axiom('Plant', 'disjoint', 'Animal'))['ok'])
print()
print('readability   :', json.dumps(ch9.readability(bad), indent=1))

> **`"No plant is a animal."`** The round trip is perfect: parse it back and you recover exactly `Plant DisjointWith Animal`. And it is still wrong English.

The round trip tests whether the *inverse function* works. It has nothing to say about whether a human would accept the output — and verbalisation exists solely so that humans will read it. **An exact metric that measures the wrong thing is still measuring the wrong thing.** This is the concrete case for keeping a judge alongside a decision procedure, and Notebook 4 scores both halves separately.

In [ ]:
print('every sample sentence, scored for readability:\n')
rows = []
for a in ch9.SAMPLE_AXIOMS:
    s = ch9.verbalise(a)
    r = ch9.readability(s)
    rows.append({'sentence': s, 'wrong article': r['wrong_article'],
                 'identifier leak': r['contains_identifier'],
                 'score': r['score']})
print(pd.DataFrame(rows).to_string(index=False))
print('\nEvery one of these round-trips perfectly.')

### Exercise 1.1 — Fix the article without breaking the round trip

Repair `"No plant is a animal."` so it reads correctly, and confirm the corrected sentence still parses back to the same axiom.

> **Hint.** `is an?` in the parser pattern already accepts both forms.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution 1.1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
import re
axiom = ch9.Axiom('Plant', 'disjoint', 'Animal')
original = ch9.verbalise(axiom)
fixed = re.sub(r'\ba (?=[aeiou])', 'an ', original, flags=re.I)
print('before:', original, '->', ch9.readability(original)['score'])
print('after :', fixed, '->', ch9.readability(fixed)['score'])

recovered = ch9.parse_cnl(fixed)
print('\nstill round-trips:', recovered is not None and recovered.key() == axiom.key())
assert recovered.key() == axiom.key()
assert ch9.readability(fixed)['score'] > ch9.readability(original)['score']
print('\nThe parser accepts \'a\' or \'an\' (the pattern is `is an?`), so the fix\n'
      'costs nothing in fidelity. That is the ideal case: a presentation problem\n'
      'that can be fixed without touching meaning. Pluralisation (\'eats only\n'
      'leaf\') is the harder cousin -- see Exercise 1.2.')

### Exercise 1.2 — Find a sentence that is faithful and unacceptable

`Every giraffe eats only leaf.` round-trips. Explain what is wrong with it, and say what the toolkit would need in order to detect the problem automatically.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution 1.2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
axiom = ch9.Axiom('Giraffe', 'only', 'Leaf', 'eats')
sentence = ch9.verbalise(axiom)
print('sentence      :', sentence)
print('round-trips   :', ch9.round_trips(axiom)['ok'])
print('readability   :', ch9.readability(sentence)['score'])
assert ch9.round_trips(axiom)['ok']
print('\nThe universal restriction reads as a bare singular: "only leaf" should\n'
      'be "only leaves". The readability check scores it 1.0 because it looks\n'
      'for articles and identifiers, not number agreement.\n\n'
      'Detecting it needs a plural form in the lexicon -- i.e. the lexicon must\n'
      'carry MORPHOLOGY, not just a string per language. That is exactly what\n'
      'lemon models and why §9.1 treats a lexicon as a structured artefact\n'
      'rather than a translation table.')